# AccioJob DA-16: Is That College Worth the Fees?
### A Comprehensive Value-for-Money (VFM) Analytics Project on NIRF 2025 Engineering Colleges

**Stakeholder:** Prospective engineering students and their families choosing an engineering college.
**Decision:** Which college offers the highest return relative to mandatory academic fees?
**Value Metric:**
$$\text{Fee-to-Salary Ratio} = \frac{\text{Total 4-Year Mandatory Academic Fees}}{\text{Annual Median Placement Salary}}$$

*Lower ratio indicates fewer years of gross salary needed to recover academic fees.*

## 1. Load Clean Merged Dataset (`colleges_final.csv`)

In [ ]:
import csv

colleges = []
with open('colleges_final.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        colleges.append(row)

print(f"Total institutions loaded: {len(colleges)}")
eligible = [c for c in colleges if c['vfm_eligible'] == 'True']
print(f"Eligible 4-Year UG Engineering institutions: {len(eligible)}")
pg_only = [c for c in colleges if c['vfm_eligible'] == 'False']
print(f"Postgraduate-only / No UG 4-Year institutions: {len(pg_only)}: {[c['college_name'] for c in pg_only]}")

## 2. Overall Cohort Statistics

In [ ]:
fees = [float(c['total_4yr_fees_inr']) / 100000.0 for c in eligible]
salaries = [float(c['median_salary_inr']) / 100000.0 for c in eligible]
ratios = [float(c['fee_salary_ratio']) for c in eligible]
placements = [float(c['placement_percentage']) for c in eligible]

def stats(arr):
    s = sorted(arr)
    n = len(s)
    mean_val = sum(s) / n
    med_val = s[n // 2] if n % 2 != 0 else (s[n // 2 - 1] + s[n // 2]) / 2.0
    return mean_val, med_val, min(s), max(s)

print(f"=== COHORT SUMMARY (N={len(eligible)}) ===")
f_mean, f_med, f_min, f_max = stats(fees)
print(f"4-Year Academic Fees: Mean = ₹{f_mean:.2f}L | Median = ₹{f_med:.2f}L | Range = ₹{f_min:.2f}L to ₹{f_max:.2f}L")

s_mean, s_med, s_min, s_max = stats(salaries)
print(f"Median Placement Salary: Mean = ₹{s_mean:.2f}LPA | Median = ₹{s_med:.2f}LPA | Range = ₹{s_min:.2f}LPA to ₹{s_max:.2f}LPA")

r_mean, r_med, r_min, r_max = stats(ratios)
print(f"Fee-to-Salary Ratio: Mean = {r_mean:.3f} yrs | Median = {r_med:.3f} yrs | Range = {r_min:.3f} yrs to {r_max:.3f} yrs")

p_mean, p_med, p_min, p_max = stats(placements)
print(f"Placement Percentage: Mean = {p_mean:.2f}% | Median = {p_med:.2f}%")

## 3. Analysis by NIRF Rank Band
Comparing ranks 1–25 with ranks 76–100 reveals the 'mid-tier trap', where private universities increase average costs without matching top-tier salaries.

In [ ]:
bands = {
    'Band 1 (Ranks 1-25)': [c for c in eligible if 1 <= int(c['nirf_rank']) <= 25],
    'Band 2 (Ranks 26-50)': [c for c in eligible if 26 <= int(c['nirf_rank']) <= 50],
    'Band 3 (Ranks 51-75)': [c for c in eligible if 51 <= int(c['nirf_rank']) <= 75],
    'Band 4 (Ranks 76-100)': [c for c in eligible if 76 <= int(c['nirf_rank']) <= 100],
}

print(f"{'Rank Band':<22} | {'N':<3} | {'Avg Fee':<10} | {'Avg Sal':<10} | {'Avg Ratio':<10} | {'Avg Placed':<10}")
print('-'*75)
for b_name, b_colleges in bands.items():
    b_fees = [float(c['total_4yr_fees_inr'])/1e5 for c in b_colleges]
    b_sals = [float(c['median_salary_inr'])/1e5 for c in b_colleges]
    b_rats = [float(c['fee_salary_ratio']) for c in b_colleges]
    b_plcs = [float(c['placement_percentage']) for c in b_colleges]
    print(f"{b_name:<22} | {len(b_colleges):<3} | ₹{sum(b_fees)/len(b_fees):5.2f}L     | ₹{sum(b_sals)/len(b_sals):5.2f}LPA   | {sum(b_rats)/len(b_rats):.3f} yrs   | {sum(b_plcs)/len(b_plcs):5.1f}%")

## 4. Top 10 Best Value Institutions (Lowest Recovery Years)
Public state and central universities offer extraordinary value, requiring only days or weeks of salary to recover 4 years of tuition.

In [ ]:
sorted_by_vfm = sorted(eligible, key=lambda c: float(c['fee_salary_ratio']))
print(f"{'Rank':<5} | {'College Name':<42} | {'State':<14} | {'Fees (L)':<9} | {'Salary (LPA)':<12} | {'Ratio (yrs)':<11} | {'Placement'}")
print('-'*105)
for c in sorted_by_vfm[:10]:
    name = c['college_name'][:40]
    print(f"#{int(c['nirf_rank']):<4} | {name:<42} | {c['state']:<14} | ₹{float(c['total_4yr_fees_inr'])/1e5:5.2f}L  | ₹{float(c['median_salary_inr'])/1e5:5.2f}LPA     | {float(c['fee_salary_ratio']):.3f} yrs     | {float(c['placement_percentage']):5.1f}%")

## 5. Top 10 Worst Value Institutions (Highest Recovery Years)
High-tuition private and deemed universities require upwards of 2 to 3 years of gross median salary to recover academic dues.

In [ ]:
print(f"{'Rank':<5} | {'College Name':<42} | {'State':<14} | {'Fees (L)':<9} | {'Salary (LPA)':<12} | {'Ratio (yrs)':<11} | {'Placement'}")
print('-'*105)
for c in sorted_by_vfm[-10:][::-1]:
    name = c['college_name'][:40]
    print(f"#{int(c['nirf_rank']):<4} | {name:<42} | {c['state']:<14} | ₹{float(c['total_4yr_fees_inr'])/1e5:5.2f}L  | ₹{float(c['median_salary_inr'])/1e5:5.2f}LPA     | {float(c['fee_salary_ratio']):.3f} yrs     | {float(c['placement_percentage']):5.1f}%")

## 6. State-by-State Value Distribution
Summarizing the financial return across India's states.

In [ ]:
states = {}
for c in eligible:
    st = c['state']
    states.setdefault(st, []).append(c)

state_summary = []
for st, clist in states.items():
    st_fees = [float(c['total_4yr_fees_inr'])/1e5 for c in clist]
    st_sals = [float(c['median_salary_inr'])/1e5 for c in clist]
    st_rats = [float(c['fee_salary_ratio']) for c in clist]
    st_plcs = [float(c['placement_percentage']) for c in clist]
    state_summary.append({
        'state': st,
        'count': len(clist),
        'avg_fee': sum(st_fees)/len(st_fees),
        'avg_sal': sum(st_sals)/len(st_sals),
        'avg_ratio': sum(st_rats)/len(st_rats),
        'avg_plc': sum(st_plcs)/len(st_plcs)
    })

state_summary.sort(key=lambda x: x['avg_ratio'])

print(f"{'State':<18} | {'N':<3} | {'Avg Fee':<10} | {'Avg Sal':<11} | {'Avg Ratio':<10} | {'Avg Placed':<10}")
print('-'*75)
for s in state_summary:
    print(f"{s['state']:<18} | {s['count']:<3} | ₹{s['avg_fee']:5.2f}L     | ₹{s['avg_sal']:5.2f}LPA    | {s['avg_ratio']:.3f} yrs   | {s['avg_plc']:5.1f}%")

## 7. Conclusions & Strategic Guidance

1. **Public vs Private Divergence:** Highly subsidized public institutions (Jadavpur, Jamia, Anna Univ, IITs, NITs) provide unparalleled capital protection.
2. **Mid-Tier Caution:** Private institutions ranked 26–75 frequently demand >₹15 Lakhs for median salaries comparable to or lower than tier-2 government colleges.
3. **Analytical Limits:** Institutional medians mask branch variation (Computer Science vs Core Engineering) and do not account for non-placed students or ₹5–₹8 Lakhs in 4-year living expenses.